In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from pandas import Timedelta
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from statsmodels.tsa.ar_model import AutoReg
import matplotlib.pyplot as plt
import copy
import random
import warnings
warnings.filterwarnings('ignore')

# Random seeds
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

In [ ]:
# LSTM Config
class Config_LSTM:
    # ═══════════════════════════════════════════════════════════════
    #  KEY PARAMETERS  ─  most likely things you will want to change
    # ═══════════════════════════════════════════════════════════════

    # BIC toggle:
    #   True  → lookback chosen each window via HQIC (AR model on val set)
    #           Output: two Excel sheets (RV predictions + chosen lookback per date)
    #   False → fixed lookback_period below used for every window & every stock
    #           Output: one Excel sheet (RV predictions only)
    use_bic: bool = False

    # ── Fixed lookback window (used when use_bic=False) ──────────────
    # 22 trading days ≈ 1 month; standard in financial-econometrics LSTM work.
    # Change to any positive integer, e.g. 5 (1 week), 10, 44 (2 months).
    lookback_period = 22

    # Maximum lag the BIC search considers (used when use_bic=True)
    max_bic_lag = 44

    # ── Expanding-window sizes ───────────────────────────────────────
    # initial_train_years → minimum history before the FIRST test window
    #                       (so first prediction appears ~initial_train_years
    #                        + validation_years after a stock's start date)
    # validation_years    → val-set length; also the gap between train-end
    #                       and test-start in each window
    # test_months         → length of each test window (= step size forward)
    initial_train_years = 5
    validation_years    = 1
    test_months         = 12

    # ═══════════════════════════════════════════════════════════════
    #  MODEL / TRAINING PARAMETERS
    # ═══════════════════════════════════════════════════════════════

    hidden_layers         = [16, 8]
    learning_rate         = 0.001
    batch_size            = 512
    epochs                = 100
    early_stopping_rounds = 10
    n_ensembles           = 10

    # Full contiguous lag list derived from lookback_period (used when use_bic=False)
    lag_candidates = sorted(list(range(1, lookback_period + 1)), reverse=True)

    # Transformation parameters
    use_log_transform = True   # model log(RV) instead of RV directly
    target_scaler     = True   # standardise the target variable

    # Device (auto-detected)
    if torch.cuda.is_available():
        device = torch.device('cuda')
        print("Using CUDA")
    elif torch.backends.mps.is_available():
        device = torch.device('mps')
        print("Using MPS")
    else:
        device = torch.device('cpu')
        print("Using CPU")

In [ ]:
class LSTMWithBatchNorm(nn.Module):
    def __init__(self, input_dim: int, hidden_layers, dropout: float = 0.0):
        super().__init__()

        self.input_dim = int(input_dim)
        self.hidden_layers = list(hidden_layers)
        self.dropout = float(dropout)

        # Build LSTM stack
        self.lstm_layers = nn.ModuleList()
        self.lstm_layers.append(nn.LSTM(
            input_size=1,
            hidden_size=self.hidden_layers[0],
            num_layers=1,
            batch_first=True
        ))

        for in_size, out_size in zip(self.hidden_layers[:-1], self.hidden_layers[1:]):
            self.lstm_layers.append(nn.LSTM(
                input_size=in_size,
                hidden_size=out_size,
                num_layers=1,
                batch_first=True
            ))

        self.use_dropout = self.dropout > 0.0
        if self.use_dropout:
            self.dropout_layer = nn.Dropout(self.dropout)

        last_hidden = self.hidden_layers[-1]
        self.bn = nn.BatchNorm1d(last_hidden)
        self.fc_out = nn.Linear(last_hidden, 1)

        nn.init.kaiming_uniform_(self.fc_out.weight, a=np.sqrt(5))
        if self.fc_out.bias is not None:
            fan_in, _ = nn.init._calculate_fan_in_and_fan_out(self.fc_out.weight)
            bound = 1 / np.sqrt(fan_in)
            nn.init.uniform_(self.fc_out.bias, -bound, bound)

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(-1)
        elif x.dim() == 3:
            pass
        else:
            raise ValueError(f"Expected input of shape (N, L) or (N, L, C), got {tuple(x.shape)}")

        for lstm in self.lstm_layers:
            x, _ = lstm(x)

        last = x[:, -1, :]
        last = self.bn(last)
        if self.use_dropout:
            last = self.dropout_layer(last)

        out = self.fc_out(last)
        return out

In [ ]:
# ── Data loading ──────────────────────────────────────────────────────────────

def load_rv_series(filepath):
    """
    Load the rv_series xlsx file.
    Column A = date, columns B onwards = realised variance per stock (ticker as header).
    Returns a DataFrame with a DatetimeIndex and one column per stock.
    Values are stored as-is (RV); log transform is applied inside the pipeline.
    """
    df = pd.read_excel(filepath, index_col=0, parse_dates=True)
    df.index = pd.to_datetime(df.index)
    df.index.name = 'date'
    df = df.sort_index()
    # Drop fully-empty columns
    df = df.dropna(axis=1, how='all')
    print(f"Loaded rv_series: {df.shape[0]} dates x {df.shape[1]} stocks")
    print(f"Global date range: {df.index.min().date()} to {df.index.max().date()}")
    return df


def get_stock_series(rv_df, ticker):
    """
    Extract a single stock's RV series, dropping leading/trailing NaNs
    but keeping the date index aligned to the global calendar.
    Returns a Series with the global DatetimeIndex (NaN where stock has no data).
    """
    s = rv_df[ticker]
    # Identify the stock's own start/end (first/last non-NaN)
    valid = s.dropna()
    if len(valid) == 0:
        raise ValueError(f"No data for ticker {ticker}")
    stock_start = valid.index[0]
    stock_end   = valid.index[-1]
    print(f"  [{ticker}] Data: {stock_start.date()} to {stock_end.date()} ({len(valid)} obs)")
    return s  # keep NaNs for global alignment


def create_lagged_features(series, lags):
    """
    Given a Series (RV values), build a DataFrame with lagged columns.
    The column 'RV_daily' holds the raw RV (target before log-transform).
    NaN rows from lagging are dropped.
    """
    df = pd.DataFrame({'RV_daily': series})
    for lag in lags:
        df[f'lag_{lag}'] = df['RV_daily'].shift(lag)
    df = df.dropna()
    return df


def prepare_train_val_test_data(df, train_start, train_end, val_end, test_end):
    train_data = df[train_start:train_end]
    val_data   = df[train_end:val_end]
    test_data  = df[val_end:test_end]
    return train_data, val_data, test_data

In [ ]:
# ── BIC lag selection ─────────────────────────────────────────────────────────

def calculate_BIC(val_data, max_lag: int = 44):
    """
    Fit AR models on val_data (log-RV) and select the best lag via HQIC.
    Returns a sorted list of lag indices (descending) for LSTM sequence input,
    and the chosen best_lag integer for recording.
    """
    bic_values = {}
    for lag in range(2, max_lag + 1):
        model  = AutoReg(val_data, lags=lag, old_names=False)
        result = model.fit()
        bic_values[lag] = result.hqic

    best_lag = min(bic_values, key=bic_values.get)

    chosen = [1]
    if best_lag > 10:
        second_var = int(best_lag * 5 / 22)
        chosen.append(second_var)
        chosen.append(best_lag)
    else:
        second_var = round(best_lag * 22 / 5)
        chosen.append(best_lag)
        chosen.append(second_var)

    # Full contiguous lag range up to the max chosen (for LSTM)
    lookback = sorted(list(range(1, chosen[-1] + 1)), reverse=True)
    print(f"    BIC-selected lookback period: {chosen[-1]} ({len(lookback)} lags)")
    return lookback, best_lag

In [ ]:
# ── Training utilities ────────────────────────────────────────────────────────

class EarlyStopping:
    def __init__(self, patience=10, min_delta=0):
        self.patience   = patience
        self.min_delta  = min_delta
        self.counter    = 0
        self.best_loss  = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter   = 0


def train_single_model(X_train, y_train, X_val, y_val, config, verbose=False):
    X_train_tensor = torch.FloatTensor(X_train).to(config.device)
    y_train_tensor = torch.FloatTensor(y_train).to(config.device)
    X_val_tensor   = torch.FloatTensor(X_val).to(config.device)
    y_val_tensor   = torch.FloatTensor(y_val).to(config.device)

    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    train_loader  = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)

    model     = LSTMWithBatchNorm(X_train.shape[1], config.hidden_layers).to(config.device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=config.learning_rate)

    early_stopping = EarlyStopping(patience=config.early_stopping_rounds)
    best_val   = float('inf')
    best_state = None
    best_epoch = 0

    for epoch in range(config.epochs):
        model.train()
        epoch_train_loss = 0.0
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            out  = model(batch_X)
            loss = criterion(out, batch_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_train_loss += loss.item()

        avg_train_loss = epoch_train_loss / max(1, len(train_loader))

        model.eval()
        with torch.inference_mode():
            vout  = model(X_val_tensor)
            vloss = criterion(vout, y_val_tensor).item()

        if (best_val - vloss) > early_stopping.min_delta:
            best_val   = vloss
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1

        early_stopping(vloss)
        if early_stopping.early_stop:
            if verbose:
                print(f"Early stopping at epoch {epoch+1}; best={best_epoch} val={best_val:.6f}")
            break

        if verbose and (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{config.epochs}  Train: {avg_train_loss:.6f}  Val: {vloss:.6f}")

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


def set_seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def train_ensemble(X_train, y_train, X_val, y_val, config):
    models    = []
    base_seed = 42
    print(f"    Training ensemble of {config.n_ensembles} models...")
    for i in range(config.n_ensembles):
        set_seed_all(base_seed + i)
        model = train_single_model(X_train, y_train, X_val, y_val, config, verbose=False)
        models.append(model)
    return models


def predict_ensemble(models, X, config):
    X_tensor    = torch.FloatTensor(X).to(config.device)
    predictions = []
    for model in models:
        model.eval()
        with torch.no_grad():
            predictions.append(model(X_tensor).cpu().numpy())
    return np.stack(predictions, axis=0)

In [ ]:
# ── Metrics ───────────────────────────────────────────────────────────────────

def calculate_mse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred)

def calculate_qlike(y_true, y_pred):
    y_true = np.maximum(y_true, 1e-8)
    y_pred = np.maximum(y_pred, 1e-8)
    return np.mean(y_true / y_pred - np.log(y_true / y_pred) - 1)

In [ ]:
# ── Per-stock expanding window forecast ───────────────────────────────────────

def expanding_window_forecast_single(
    ticker: str,
    rv_series: pd.Series,   # raw RV values, NaN outside stock's history
    global_index: pd.DatetimeIndex,
    config
):
    """
    Run the LSTM expanding-window forecast for ONE stock.

    Returns
    -------
    pred_series : pd.Series  indexed on global_index (NaN where no prediction)
    bic_series  : pd.Series  indexed on global_index (NaN where not applicable)
                  – contains the chosen lookback period per test date when use_bic=True
                  – all NaN when use_bic=False
    """
    # ── build lagged feature frame for this stock ──────────────────────────
    stock_rv = rv_series.dropna()          # restrict to the stock's own date range

    if config.use_bic:
        df_with_lags = create_lagged_features(stock_rv, list(range(1, config.max_bic_lag + 1)))
    else:
        df_with_lags = create_lagged_features(stock_rv, config.lag_candidates)
        feature_cols = [col for col in df_with_lags.columns if col.startswith('lag_')]

    # Pre-log-transform the RV column used for BIC fitting
    df_with_lags['log_RV'] = np.log(df_with_lags['RV_daily'].clip(lower=1e-8))

    # ── expanding window setup ─────────────────────────────────────────────
    start_date        = df_with_lags.index[0]
    initial_train_end = start_date + pd.DateOffset(years=config.initial_train_years)
    current_test_start = initial_train_end + pd.DateOffset(years=config.validation_years)

    # Storage: one value per global date
    pred_dict = {}   # date -> predicted RV
    bic_dict  = {}   # date -> chosen lookback period (int)

    window_count = 0

    while current_test_start < df_with_lags.index[-1]:
        window_count += 1

        train_end  = current_test_start - pd.DateOffset(years=config.validation_years)
        val_start  = train_end + Timedelta(days=1)
        val_end    = current_test_start + Timedelta(days=1)
        test_end   = min(
            current_test_start + pd.DateOffset(months=config.test_months),
            df_with_lags.index[-1]
        ) + Timedelta(days=2)

        train_data = df_with_lags[start_date:train_end]
        val_data   = df_with_lags[val_start:val_end]
        test_data  = df_with_lags[val_end + Timedelta(days=1):test_end]

        if len(test_data) == 0 or len(train_data) == 0 or len(val_data) == 0:
            break

        print(f"  [{ticker}] Window {window_count}: "
              f"train {start_date.date()}–{train_end.date()} "
              f"| val {val_start.date()}–{val_end.date()} "
              f"| test {(val_end + Timedelta(days=1)).date()}–{test_end.date()} "
              f"({len(test_data)} obs)")

        # ── lag selection ──────────────────────────────────────────────────
        if config.use_bic:
            bic_lags, best_lag_int = calculate_BIC(val_data['log_RV'], config.max_bic_lag)
            current_feature_cols   = [f'lag_{lag}' for lag in bic_lags]
        else:
            current_feature_cols = feature_cols
            best_lag_int         = None

        # ── features & targets ─────────────────────────────────────────────
        X_train = train_data[current_feature_cols].values
        X_val   = val_data[current_feature_cols].values
        X_test  = test_data[current_feature_cols].values

        y_train_original = train_data['RV_daily'].values.reshape(-1, 1)
        y_val_original   = val_data['RV_daily'].values.reshape(-1, 1)
        y_test_original  = test_data['RV_daily'].values.reshape(-1, 1)

        if config.use_log_transform:
            y_train = np.log(np.maximum(y_train_original, 1e-8))
            y_val   = np.log(np.maximum(y_val_original,   1e-8))
        else:
            y_train = y_train_original.copy()
            y_val   = y_val_original.copy()

        # ── scaling ────────────────────────────────────────────────────────
        feature_scaler  = StandardScaler()
        X_train_scaled  = feature_scaler.fit_transform(X_train)
        X_val_scaled    = feature_scaler.transform(X_val)
        X_test_scaled   = feature_scaler.transform(X_test)

        if config.target_scaler:
            target_scaler   = StandardScaler()
            y_train_scaled  = target_scaler.fit_transform(y_train)
            y_val_scaled    = target_scaler.transform(y_val)
        else:
            y_train_scaled  = y_train
            y_val_scaled    = y_val
            target_scaler   = None

        # ── train & predict ────────────────────────────────────────────────
        models = train_ensemble(X_train_scaled, y_train_scaled, X_val_scaled, y_val_scaled, config)

        test_pred_scaled_all = predict_ensemble(models, X_test_scaled, config)

        if config.target_scaler and target_scaler is not None:
            test_log_all = np.stack(
                [target_scaler.inverse_transform(p) for p in test_pred_scaled_all], axis=0
            )
        else:
            test_log_all = test_pred_scaled_all

        if config.use_log_transform:
            test_pred = np.mean(np.exp(test_log_all), axis=0)
        else:
            test_pred = np.mean(test_log_all, axis=0)

        test_pred = np.maximum(test_pred, 1e-12).flatten()

        # ── store predictions aligned to global index ──────────────────────
        for i, date in enumerate(test_data.index):
            pred_dict[date] = test_pred[i]
            if config.use_bic:
                bic_dict[date] = best_lag_int   # same lookback chosen for whole test window

        current_test_start = current_test_start + pd.DateOffset(months=config.test_months)

    # ── reindex to global calendar ─────────────────────────────────────────
    pred_series = pd.Series(pred_dict, name=ticker).reindex(global_index)
    bic_series  = pd.Series(bic_dict,  name=ticker).reindex(global_index)

    return pred_series, bic_series

In [ ]:
# ── Run all stocks (sequential, GPU-friendly) ─────────────────────────────────

def run_all_stocks(rv_df: pd.DataFrame, config) -> tuple:
    """
    Iterate over every stock column in rv_df and run the expanding-window LSTM.

    Returns
    -------
    pred_df : DataFrame  shape (dates, stocks) – predicted RV, NaN where no prediction
    bic_df  : DataFrame  shape (dates, stocks) – chosen lookback period (use_bic=True only)
    """
    global_index = rv_df.index
    tickers      = rv_df.columns.tolist()

    pred_cols = {}
    bic_cols  = {}

    for ticker in tickers:
        print(f"\n{'='*60}")
        print(f"Processing: {ticker}")
        print(f"{'='*60}")
        try:
            pred_s, bic_s = expanding_window_forecast_single(
                ticker      = ticker,
                rv_series   = rv_df[ticker],
                global_index= global_index,
                config      = config
            )
            pred_cols[ticker] = pred_s
            bic_cols[ticker]  = bic_s
        except Exception as e:
            print(f"  [{ticker}] FAILED: {e}")
            pred_cols[ticker] = pd.Series(np.nan, index=global_index, name=ticker)
            bic_cols[ticker]  = pd.Series(np.nan, index=global_index, name=ticker)

    pred_df = pd.DataFrame(pred_cols, index=global_index)
    bic_df  = pd.DataFrame(bic_cols,  index=global_index)

    return pred_df, bic_df

In [ ]:
# ── Main entry point ──────────────────────────────────────────────────────────

def main_lstm():
    print("="*60)
    print("LSTM NEURAL NETWORK – ALL STOCKS")
    print("="*60)

    config = Config_LSTM()
    print(f"\nUsing device: {config.device}")
    print(f"BIC enabled:  {config.use_bic}")

    # ── Load data ──────────────────────────────────────────────────────────
    print("\nLoading rv_series.xlsx ...")
    rv_df = load_rv_series('../Data/rv_series.xlsx')   # <-- update filename if needed

    # ── Run forecasts ──────────────────────────────────────────────────────
    pred_df, bic_df = run_all_stocks(rv_df, config)

    # ── Save output ────────────────────────────────────────────────────────
    if config.use_bic:
        output_filename = 'LSTM_BIC_predictions.xlsx'
        with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
            pred_df.to_excel(writer, sheet_name='RV_Predictions')
            bic_df.to_excel(writer,  sheet_name='BIC_Lookback_Period')
        print(f"\nSaved predictions + BIC lookback periods to '{output_filename}'")
        print("  Sheet 1: RV_Predictions  (predicted realised variance per stock)")
        print("  Sheet 2: BIC_Lookback_Period  (chosen lookback window per test date)")
    else:
        output_filename = 'LSTM_predictions.xlsx'
        with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
            pred_df.to_excel(writer, sheet_name='RV_Predictions')
        print(f"\nSaved predictions to '{output_filename}'")

    print(f"\nOutput shape: {pred_df.shape[0]} dates x {pred_df.shape[1]} stocks")
    non_nan = pred_df.notna().sum().sum()
    print(f"Total predictions made: {non_nan:,}")

    return pred_df, bic_df


if __name__ == '__main__':
    pred_df, bic_df = main_lstm()